In [0]:
from collections import defaultdict
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import Dataset, DataLoader, IterableDataset
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, ArrayType, StringType
from pyspark.ml.feature import StringIndexer
import pyarrow.parquet as pq
from itertools import chain
import pickle
import glob
import os
import shutil
import gc
import json
from time import perf_counter

In [0]:
def resolve_local_result_dir(path_str):
    """Convert dbfs:/ paths to local driver file-system paths."""
    if path_str.startswith("dbfs:/"):
        return "/dbfs/" + path_str[len("dbfs:/"):].lstrip("/")
    return path_str


def apply_shard_filter(df, shard_id, num_shards):
    """Filter a dataframe to the current shard based on shard_id or stable hash fallback."""
    if int(num_shards) <= 1:
        return df

    if "shard_id" in df.columns:
        return df.where(F.col("shard_id") == F.lit(int(shard_id)))

    # Fallback for legacy tables without shard_id column
    return df.where(F.pmod(F.xxhash64("user_id"), F.lit(int(num_shards))) == F.lit(int(shard_id)))


def apply_partition_date_filter_if_exists(df, run_date):
    """Apply partition_date filter only for tables that contain that column."""
    if "partition_date" in df.columns:
        return df.where(F.col("partition_date") == run_date)
    return df


def normalize_ranking_shard_id(raw_shard_id, num_shards, shard_id_base):
    """Normalize pipeline shard ids to the zero-based ids used by the Spark shard column."""
    if int(num_shards) < 1:
        raise ValueError(f"ranking_num_shards must be >= 1, got {num_shards}")

    shard_id_base = str(shard_id_base or "0").strip().lower()
    if shard_id_base not in {"0", "1"}:
        raise ValueError(
            f"ranking_shard_id_base must be '0' or '1', got {shard_id_base}"
        )

    if shard_id_base == "0":
        if raw_shard_id < 0 or raw_shard_id >= num_shards:
            raise ValueError(
                f"ranking_shard_id must be in [0, {num_shards - 1}] when ranking_shard_id_base=0, got {raw_shard_id}"
            )
        return raw_shard_id

    if raw_shard_id < 1 or raw_shard_id > num_shards:
        raise ValueError(
            f"ranking_shard_id must be in [1, {num_shards}] when ranking_shard_id_base=1, got {raw_shard_id}"
        )
    return raw_shard_id - 1

# -----------------------------
# Dataset construction
# -----------------------------
def load_user_post_spark_dataframes(spark, db, table_names, run_date):
    """Load inference dataframes for a specific date."""
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['user_processed_features']}
        WHERE partition_date = '{run_date}'
    """)

    post_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['post_processed_features']}
        WHERE partition_date = '{run_date}'
    """)

    # # Cache since we'll use them later
    # user_df.cache()
    # post_df.cache()

    return user_df, post_df

def build_shard_retrieval_dataframe(spark, db, table_names, run_date, shard_id=0, num_shards=1):
    retrieval_base_df = spark.table(f"{db}.{table_names['all_retrieval_results']}")
    retrieval_base_df = apply_partition_date_filter_if_exists(retrieval_base_df, run_date)
    retrieval_base_df = apply_shard_filter(retrieval_base_df, shard_id, num_shards)

    # Build shard-local row index once so chunk boundaries are correct inside each shard.
    retrieval_base_df = retrieval_base_df.withColumn(
        "__local_row_id",
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id()))
    )
    return retrieval_base_df


def load_retrieval_spark_dataframes(retrieval_base_df, chunk_start_idx, chunk_size):
    start_row = int(chunk_start_idx) + 1
    end_row = int(chunk_start_idx + chunk_size)
    return retrieval_base_df.where(
        (F.col("__local_row_id") >= F.lit(start_row))
        & (F.col("__local_row_id") <= F.lit(end_row))
    ).drop("__local_row_id")

def process_inference_dataframe(df, id_col, cat_cols, num_cols, emb_col):
    """
    Process inference data using fitted pipeline.

    Args:
        df: Input dataframe
        id_col: ID column name
        cat_cols: List of categorical column names
        num_cols: List of numerical column names
        emb_col: Embedding column name

    Returns:
        Processed dataframe and ID decoder mapping
    """
    df_proc = df
    # Create temporary integer encoding for IDs (for tensor compatibility)
    # This is different from the model input encoding - it's just for indexing

    indexer = StringIndexer(
        inputCol=id_col,
        outputCol=id_col + "_temp_idx",
        handleInvalid="keep"   # avoid failures if new IDs appear
    )

    # Fit the indexer (distributed operation)
    model = indexer.fit(df)
    # Transform the dataframe
    df_proc = model.transform(df)

    # Extract decoder mapping: index → original ID
    # labelsArray is an ordered list: index -> string
    labels = model.labels  # already collected efficiently
    decode_id_map = {i: lbl for i, lbl in enumerate(labels)}

    # Select final columns
    select_cols = [
        id_col,
        id_col + '_temp_idx',
        id_col + "_idx",
        *[f"{c}_idx" for c in cat_cols],
        *[f"{c}_norm" for c in num_cols],
        emb_col
    ]

    df_proc = df_proc.select(*select_cols)

    return df_proc, decode_id_map

def process_user_post_feature_data(user_df, post_df,
                                   user_id_col, post_id_col,
                                   user_cat_cols, user_num_cols,
                                   post_cat_cols, post_num_cols,
                                   user_emb_col, post_emb_col):
    # Process user relavant columns
    user_df_proc, decode_uid_map = process_inference_dataframe(
        user_df,
        user_id_col,
        user_cat_cols,
        user_num_cols,
        user_emb_col
    )
    print("User table process completed")

    # Process post relavant columns
    post_df_proc, decode_pid_map = process_inference_dataframe(
        post_df,
        post_id_col,
        post_cat_cols,
        post_num_cols,
        post_emb_col
    )
    print("Post table process completed")

    # Get statistics (useful for validation)
    n_users = user_df_proc.select(F.countDistinct(user_id_col + "_temp_idx")).first()[0]
    n_posts = post_df_proc.select(F.countDistinct(post_id_col + "_temp_idx")).first()[0]

    return {
        'n_users': n_users,
        'n_posts': n_posts,
        'user_df': user_df_proc,
        'post_df': post_df_proc,
        'uid_decoder': decode_uid_map,
        'pid_decoder': decode_pid_map
    }

def join_ranking_inference_data(user_df, post_df, retrieval_df, user_id_col, post_id_col):
    """
    Join retrieval data and feature data for inference
    """
    # Repartition
    retrieval_df = retrieval_df.repartition(user_id_col, post_id_col)
    user_df = user_df.repartition(user_id_col)
    post_df = post_df.repartition(post_id_col)

    # Optimized join: broadcast smaller table if possible
    # PySpark will auto-broadcast if one side is small enough
    joined_df = (retrieval_df
                 .join(user_df, user_id_col, "inner")
                 .join(post_df, post_id_col, "inner"))

    return joined_df

def get_row_cnt(retrieval_base_df):
    return retrieval_base_df.count()


In [0]:
def load_and_merge_results(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    res_files = sorted(glob.glob(f"{result_dir}/result_chunk_*.pkl"))
    results = [pickle.load(open(f, 'rb')) for f in res_files]
    merged_res = {key: [d[key] for d in results] for key in results[0]}

    return merged_res

def load_decoders(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    uid_decoder = pickle.load(open(Path(result_dir) / "uid_decoder.pkl", 'rb'))
    pid_decoder = pickle.load(open(Path(result_dir) / "pid_decoder.pkl", 'rb'))
    return uid_decoder, pid_decoder

def clear_result_dir(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    result_path = Path(result_dir)
    if not result_path.exists():
        print(f"Result directory does not exist yet: {result_dir}")
        return

    removed_entries = 0
    for child in result_path.iterdir():
        try:
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink(missing_ok=True)
            removed_entries += 1
        except OSError as e:
            print(f"Warning: could not delete {child}: {e}")

    print(f"Cleared {removed_entries} existing result artifacts from {result_dir}")

In [0]:
class PandasRankingDataset(IterableDataset):
    def __init__(self, data_df,
                 user_id_int_col, post_id_int_col,
                 user_id_col, post_id_col,
                 user_cat_cols, user_num_cols,
                 post_cat_cols, post_num_cols,
                 user_emb_col, post_emb_col,
                 batch_size=1024):
        super().__init__()
        self.data_df = data_df
        self.user_id_col, self.post_id_col = user_id_col, post_id_col
        self.user_id_int_col, self.post_id_int_col = user_id_int_col, post_id_int_col
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.post_cat_cols, self.post_num_cols = post_cat_cols, post_num_cols
        self.user_emb_col, self.post_emb_col = user_emb_col, post_emb_col
        self.batch_size = batch_size

    def __iter__(self):
        # Iterate over the Pandas DataFrame in chunks
        num_rows = len(self.data_df)

        for i in range(0, num_rows, self.batch_size):
            # Slice the Pandas DataFrame to get a batch
            batch = self.data_df.iloc[i : i + self.batch_size]

            # Convert columns to tensors
            # Note: .values is faster than .to_numpy() for simple columns
            user_id = torch.tensor(batch[self.user_id_col].values, dtype=torch.long)
            user_id_int = torch.tensor(batch[self.user_id_int_col].values, dtype=torch.long)

            post_id_int = torch.tensor(batch[self.post_id_int_col].values, dtype=torch.long)
            post_id = torch.tensor(batch[self.post_id_col].values, dtype=torch.long)

            # Categorical features
            user_cat = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.long)
                for col in self.user_cat_cols
            ], dim=1) if self.user_cat_cols else torch.empty((user_id.size(0), 0), dtype=torch.long)

            # Numerical features
            user_num = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.float32)
                for col in self.user_num_cols
            ], dim=1) if self.user_num_cols else torch.empty((user_id.size(0), 0), dtype=torch.float32)

            post_cat = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.long)
                for col in self.post_cat_cols
            ], dim=1) if self.post_cat_cols else torch.empty((post_id.size(0), 0), dtype=torch.long)

            post_num = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.float32)
                for col in self.post_num_cols
            ], dim=1) if self.post_num_cols else torch.empty((post_id.size(0), 0), dtype=torch.float32)

            # Process User Embeddings
            # In Pandas, ArrayType columns become numpy arrays of objects/lists
            user_emb_list = batch[self.user_emb_col].values
            emb_dim = len(user_emb_list[0]) if len(user_emb_list) > 0 else 512

            # Fast vectorized padding/stacking is hard with object arrays,
            # so we keep the list comprehension but optimized
            user_padded_embs = []
            for emb in user_emb_list:
                if len(emb) == emb_dim:
                    user_padded_embs.append(emb)
                elif len(emb) > emb_dim:
                    user_padded_embs.append(emb[:emb_dim])
                else:
                    user_padded_embs.append(np.pad(emb, (0, emb_dim - len(emb)), mode='constant'))

            lastn_embs = torch.tensor(np.stack(user_padded_embs), dtype=torch.float32)

            # Process Post Embeddings
            post_emb_list = batch[self.post_emb_col].values
            emb_dim_post = len(post_emb_list[0]) if len(post_emb_list) > 0 else 512

            post_padded_embs = []
            for emb in post_emb_list:
                if len(emb) == emb_dim_post:
                    post_padded_embs.append(emb)
                elif len(emb) > emb_dim_post:
                    post_padded_embs.append(emb[:emb_dim_post])
                else:
                    post_padded_embs.append(np.pad(emb, (0, emb_dim_post - len(emb)), mode='constant'))

            post_emb = torch.tensor(np.stack(post_padded_embs), dtype=torch.float32)

            yield {
                "user_id": user_id,
                "user_id_int": user_id_int,
                "user_cat": user_cat,
                "user_num": user_num,
                "lastn_embs": lastn_embs,
                "post_id": post_id,
                "post_id_int": post_id_int,
                "post_cat": post_cat,
                "post_num": post_num,
                "post_emb": post_emb
            }

In [0]:
def create_data_loader(data_source, user_id_col, post_id_col,
                       user_cat_cols, user_num_cols,
                       post_cat_cols, post_num_cols,
                       user_emb_col, post_emb_col,
                       batch_size):
    """
    Args:
        data_source: A Pandas DataFrame containing the chunk of data.
    """
    dataset = PandasRankingDataset(
        data_df=data_source,
        user_id_int_col=user_id_col+'_temp_idx',
        post_id_int_col=post_id_col+'_temp_idx',
        user_id_col=user_id_col+'_idx',
        post_id_col=post_id_col+'_idx',
        user_cat_cols=[c+'_idx' for c in user_cat_cols],
        user_num_cols=[c+'_norm' for c in user_num_cols],
        post_cat_cols=[c+'_idx' for c in post_cat_cols],
        post_num_cols=[c+'_norm' for c in post_num_cols],
        user_emb_col=user_emb_col,
        post_emb_col=post_emb_col,
        batch_size=batch_size
    )

    # Num_workers must be 0 for simple Pandas iteration in main process
    data_loader = DataLoader(dataset, batch_size=None, num_workers=0)
    return data_loader

In [0]:
# -----------------------------
# Model: shared trunk + heads
# -----------------------------
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(256,128), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

def calibrate_probability(p_pred, alpha: float):
    """
    Calibrate probability after training based on negative downsampling.
    Args:
        p_pred: raw sigmoid probability (tensor)
        alpha: global downsampling ratio (0 < alpha < 1)
    Returns:
        Calibrated probability tensor
    """
    return (alpha * p_pred) / ((1 - p_pred) + alpha * p_pred)

class MultiTaskRanker(nn.Module):
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim, lastn_emb_dim,
                 pid_vocab, pid_dim, post_cat_dims, post_num_dim, post_emb_dim,
                 hidden_dims=(256,128), dropout=0.2, alpha=1):
        super().__init__()
        # Embedding for user ID (with padding index 0)
        self.user_id_emb = nn.Embedding(uid_vocab+1, uid_dim, padding_idx=0)
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in user_cat_dims])

        # Embedding for post ID (with padding index 0)
        self.post_id_emb = nn.Embedding(pid_vocab+1, pid_dim, padding_idx=0)
        # Embeddings for each categorical post feature
        self.post_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in post_cat_dims])

        u_dim = uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim + lastn_emb_dim
        p_dim = pid_dim + sum(d for _, d in post_cat_dims) + post_num_dim + post_emb_dim

        # MLP to combine all features into a single vector
        self.mlp = MLP(u_dim+p_dim, hidden_dims, dropout)

        last_size = _last_linear_out_features(self.mlp.net)
        self.head_click = nn.Linear(last_size, 1)
        self.head_like = nn.Linear(last_size, 1)
        self.head_comment = nn.Linear(last_size, 1)
        self.head_share = nn.Linear(last_size, 1)
        # dwell head predicts z, with exp(z) ≈ dwell_time
        self.head_dwell = nn.Linear(last_size, 1)

        self.alpha = alpha  # global downsampling ratio

    def forward(self, user_id, user_cat, user_num, lastn_embs, post_id, post_cat, post_num, post_emb, output_type='logit'):
        # Embed user IDs
        u_id = self.user_id_emb(user_id)
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            u_cat = torch.cat([emb(user_cat[:, i]) for i, emb in enumerate(self.user_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)

        # Embed post ID
        p_id = self.post_id_emb(post_id)
        # Embed and concatenate all categorical features (if any)
        if self.post_cat_embs:
            p_cat = torch.cat([emb(post_cat[:, i]) for i, emb in enumerate(self.post_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)

        # Concatenate all features and pass through MLP
        h = self.mlp(torch.cat([u_id, u_cat, user_num, lastn_embs, p_id, p_cat, post_num, post_emb], dim=-1))

        if output_type == 'logit':
            # Raw logit outputs
            click_pred = self.head_click(h).squeeze(-1)
            like_pred = self.head_like(h).squeeze(-1)
            comment_pred = self.head_comment(h).squeeze(-1)
            share_pred = self.head_share(h).squeeze(-1)
            # Dwell time: transformation base on [P Covington, et.al. Deep Neural Networks for YouTube Recommendations. In RecSys, 2016]
            dwell_pred = self.head_dwell(h).squeeze(-1)
        elif output_type == 'prob':
            # Raw probablity outputs
            click_pred = torch.sigmoid(self.head_click(h).squeeze(-1))
            like_pred = torch.sigmoid(self.head_like(h).squeeze(-1))
            comment_pred = torch.sigmoid(self.head_comment(h).squeeze(-1))
            share_pred = torch.sigmoid(self.head_share(h).squeeze(-1))
            dwell_pred = torch.sigmoid(self.head_dwell(h).squeeze(-1))
        elif output_type == 'calibrated_prob':
            # Apply calibration to probablities when downsampling applied to imbalanced class
            # Xinran He et al. Practical lessons from predicting clicks on ads at Facebook. In the 8th International Workshop on Data Mining for Online Advertising.
            click_pred = calibrate_probability(torch.sigmoid(self.head_click(h).squeeze(-1)), self.alpha)
            like_pred = calibrate_probability(torch.sigmoid(self.head_like(h).squeeze(-1)), self.alpha)
            comment_pred = calibrate_probability(torch.sigmoid(self.head_comment(h).squeeze(-1)), self.alpha)
            share_pred = calibrate_probability(torch.sigmoid(self.head_share(h).squeeze(-1)), self.alpha)
            dwell_pred = torch.sigmoid(self.head_dwell(h).squeeze(-1))  # no calibration for dwell time
        else:
            ValueError("Output_type must be logit, prob or calibrated_prob")

        return {
            "click": click_pred,
            "like": like_pred,
            "comment": comment_pred,
            "share": share_pred,
            "dwell": dwell_pred,
        }

In [0]:
def model_inference(model, data_loader, weights, output_type='logit', device="cpu"):
    """Top-K recommendation function"""
    model.eval()

    user_ids = []
    post_ids = []
    all_scores = []

    with torch.no_grad():
        for batch in data_loader:
            inputs = (
                batch['user_id'].to(device),
                batch['user_cat'].to(device),
                batch['user_num'].to(device),
                batch['lastn_embs'].to(device),
                batch['post_id'].to(device),
                batch['post_cat'].to(device),
                batch['post_num'].to(device),
                batch['post_emb'].to(device)
            )
            preds = model(*inputs, output_type=output_type)
            if output_type == 'logit':
                # convert to probablity outputs
                preds_click= torch.sigmoid(preds["click"].squeeze())
                preds_like = torch.sigmoid(preds["like"].squeeze())
                preds_share = torch.sigmoid(preds["share"].squeeze())
                preds_comment = torch.sigmoid(preds["comment"].squeeze())
            else:
                # assume probablity outputs
                preds_click = preds["click"].squeeze()
                preds_like = preds["like"].squeeze()
                preds_share = preds["share"].squeeze()
                preds_comment = preds["comment"].squeeze()

            # convert to estimated dwell time
            preds_dwell = torch.exp(preds["dwell"].squeeze())
            # squashed estimated dwell time to [0,1] using sigmoid
            preds_dwell_norm = torch.sigmoid(preds_dwell)

            scores = (
                    weights.get('click', 1)  * preds_click.cpu().numpy()
                    + weights.get('like', 1)  * preds_like.cpu().numpy()
                    + weights.get('share', 1) * preds_share.cpu().numpy()
                    + weights.get('comment', 1) * preds_comment.cpu().numpy()
                    + weights.get('dwell', 1) * preds_dwell_norm.cpu().numpy()
            )
            # Ensure it has shape (1,) if it's a scalar
            if scores.ndim == 0:
                scores = np.array([scores])

            user_ids.append(batch['user_id_int'].cpu())
            post_ids.append(batch['post_id_int'].cpu())
            # user_ids.append(batch['user_id_int'].cpu().numpy())
            # post_ids.append(batch['post_id_int'].cpu().numpy())
            all_scores.append(scores)

    # Concatenate - handle empty case gracefully
    if not user_ids:
        return {'user_int_id': torch.empty(0, dtype=torch.long), 'post_int_id': torch.empty(0, dtype=torch.long), 'score': np.empty(0, dtype=np.float32)}

    user_ids = torch.cat(user_ids, dim=0)
    post_ids = torch.cat(post_ids, dim=0)
    # user_ids = np.concatenate(user_ids)
    # post_ids = np.concatenate(post_ids)
    y_scores = np.concatenate(all_scores)

    return {'user_int_id':user_ids, 'post_int_id':post_ids, 'score':y_scores}

In [0]:
def run_streaming_inference(spark, db, table_names, run_date,
                            data_args, model, weights, chunk_size,
                            result_dir, batch_size=1024, shard_id=0, num_shards=1):
    """Complete streaming inference pipeline without disk I/O for chunks"""

    result_dir = resolve_local_result_dir(result_dir)
    Path(result_dir).mkdir(parents=True, exist_ok=True)

    # Load data
    user_df, post_df = load_user_post_spark_dataframes(
        spark, db, table_names, run_date
    )

    proc_res = process_user_post_feature_data(user_df, post_df, **data_args)
    user_df, post_df = proc_res['user_df'], proc_res['post_df']

    retrieval_base_df = build_shard_retrieval_dataframe(
        spark,
        db,
        table_names,
        run_date,
        shard_id=shard_id,
        num_shards=num_shards,
    )

    n_rows = get_row_cnt(retrieval_base_df)

    print("\n" + "="*60)
    print("Data processing:")
    print("="*60)
    print(f"Users (inference):           {proc_res['n_users']}")
    print(f"Posts (inference):           {proc_res['n_posts']}")
    print(f"Total rows (inference):      {n_rows}")
    print("\n" + "="*60)

    # Save decoders
    pickle.dump(proc_res['uid_decoder'], open(Path(result_dir) / "uid_decoder.pkl", 'wb'))
    pickle.dump(proc_res['pid_decoder'], open(Path(result_dir) / "pid_decoder.pkl", 'wb'))
    del proc_res

    n_chunks = max(1, (n_rows + chunk_size - 1) // chunk_size)
    pipeline_started_at = perf_counter()

    for i, chunk_start_idx in enumerate(range(0, n_rows, chunk_size)):
        chunk_started_at = perf_counter()
        print(f"Run inference for chunk: {i + 1}/{n_chunks}")

        # 1. Load retrieval chunk in Spark
        retrieval_chunk_df = load_retrieval_spark_dataframes(retrieval_base_df, chunk_start_idx, chunk_size)

        # 2. Join with features in Spark
        joined_chunk_df = join_ranking_inference_data(user_df, post_df, retrieval_chunk_df, data_args['user_id_col'], data_args['post_id_col'])

        # 3. Collect to Driver (Pandas)
        collect_started_at = perf_counter()
        pandas_chunk_df = joined_chunk_df.toPandas()
        collect_elapsed = perf_counter() - collect_started_at

        # 4. Create Loader from Memory
        data_loader = create_data_loader(data_source=pandas_chunk_df, batch_size=batch_size, **data_args)

        # 5. Run inference
        inference_started_at = perf_counter()
        chunk_res = model_inference(model, data_loader, weights, device="cpu")
        inference_elapsed = perf_counter() - inference_started_at

        # 6. Save results
        pickle.dump(chunk_res, open(Path(result_dir) / f"result_chunk_{i}.pkl", 'wb'))

        chunk_elapsed = perf_counter() - chunk_started_at
        avg_chunk_elapsed = (perf_counter() - pipeline_started_at) / max(1, i + 1)
        remaining_chunks = n_chunks - (i + 1)
        estimated_remaining_minutes = (avg_chunk_elapsed * remaining_chunks) / 60.0
        print(
            f"  rows={len(pandas_chunk_df)} collect_sec={collect_elapsed:.1f} "
            f"infer_sec={inference_elapsed:.1f} total_sec={chunk_elapsed:.1f} "
            f"eta_min={estimated_remaining_minutes:.1f}"
        )

        # Cleanup memory
        del retrieval_chunk_df
        del joined_chunk_df
        del pandas_chunk_df
        del chunk_res
        del data_loader
        # gc.collect()

    return None

In [0]:
# Get pipeline parameters
dbutils.widgets.text("ranking_num_shards", "1")
dbutils.widgets.text("ranking_shard_id", "0")
dbutils.widgets.text("ranking_shard_id_base", "0")
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

ranking_num_shards = int(dbutils.widgets.get("ranking_num_shards") or "1")
ranking_shard_id_raw = int(dbutils.widgets.get("ranking_shard_id") or "0")
ranking_shard_id_base = dbutils.widgets.get("ranking_shard_id_base") or "0"
ranking_shard_id = normalize_ranking_shard_id(
    ranking_shard_id_raw,
    ranking_num_shards,
    ranking_shard_id_base,
)
print(f"Input parameter ranking_num_shards:{ranking_num_shards}")
print(f"Input parameter ranking_shard_id_raw:{ranking_shard_id_raw}")
print(f"Input parameter ranking_shard_id_base:{ranking_shard_id_base}")
print(f"Normalized ranking_shard_id:{ranking_shard_id}")

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
# config parameters
data_args = dict(
    user_id_col='user_id',
    post_id_col='post_id',
    user_cat_cols=["ip_province", "vehicle_series", "platform"],
    user_num_cols=["months_since_registration", "months_since_consent", "identity",
                   "is_koc", "has_profile_photo", "community_visits", "posts_published",
                   "posts_viewed", "posts_liked", "posts_commented", "posts_shared",
                   "users_followed", "tribes_joined"],
    post_cat_cols=["post_type"],
    post_num_cols=["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
                   "views", "likes", "comments", "shares", "dwell_time"],
    user_emb_col='lastn_embs',
    post_emb_col='post_content_emb'
)

model_args = model_config['model_args']

ranking_metadata_dir = model_config['RANKING_METADATA_PATH']
retrieval_metadata_dir = model_config['TWO_TOWER_METADATA_PATH']
result_root_dir = resolve_local_result_dir(model_config['RANKING_RESULT_PATH'])
result_dir = result_root_dir
if ranking_num_shards > 1:
    result_dir = str(Path(result_dir) / f"shard_{ranking_shard_id}")

neg_ratio = model_config['neg_sample_ratio']
batch_size = model_config['batch_size']
weights = model_config['task_weights']

chunk_size = model_config['input_data_chunk_size']

In [0]:
metadata = pickle.load(open(Path(retrieval_metadata_dir) / 'metadata_stats.pkl', 'rb'))
user_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['user_cat_dims']]
post_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['post_cat_dims']]
model = MultiTaskRanker(uid_vocab=metadata["n_users"],
                        uid_dim=model_args['id_emb_dim'],
                        user_cat_dims=user_cat_dims,
                        user_num_dim=metadata['user_num_dim'],
                        lastn_emb_dim=model_args['emb_dim'],
                        pid_vocab=metadata["n_posts"],
                        pid_dim=model_args['id_emb_dim'],
                        post_cat_dims=post_cat_dims,
                        post_num_dim=metadata['post_num_dim'],
                        post_emb_dim=model_args['emb_dim'],
                        hidden_dims=model_args['hidden_dims'],
                        dropout=model_args['dropout'],
                        alpha=neg_ratio)

model.load_state_dict(torch.load(Path(ranking_metadata_dir).joinpath("ranking_model"), weights_only=True))
model.eval()

In [0]:
# clean all existing ranking result artifacts from the volume before running the pipeline
# In parallel mode, clean only the current shard folder to avoid wiping sibling shard artifacts.
# In single-shard mode, clean the full root as before.
if ranking_num_shards > 1:
    clear_result_dir(result_dir)
else:
    clear_result_dir(result_root_dir)

In [0]:
run_streaming_inference(
    spark,
    db,
    table_names,
    run_date,
    data_args,
    model,
    weights,
    chunk_size,
    result_dir,
    batch_size,
    shard_id=ranking_shard_id,
    num_shards=ranking_num_shards,
)

In [0]:
# ============================================================
# Write result
# ============================================================
def transform_dict(result_dict, K):
    # ---- 1. Flatten all tensors/arrays ----
    user_all = torch.cat(result_dict["user_int_id"]).numpy()
    post_all = torch.cat(result_dict["post_int_id"]).numpy()
    score_all = np.concatenate(result_dict["score"])

    # ---- 2. Create distinct lists ----
    unique_users, user_inverse = np.unique(user_all, return_inverse=True)
    unique_posts, post_inverse = np.unique(post_all, return_inverse=True)

    # ---- 3. Group score+post by user ----
    user_to_scores = defaultdict(list)
    user_to_postidx = defaultdict(list)

    for i, u_idx in enumerate(user_inverse):
        user_to_scores[u_idx].append(score_all[i])
        user_to_postidx[u_idx].append(post_inverse[i])

    # ---- 4. Compute top-K scores + indices per user ----
    topk_scores = []
    topk_indices = []

    for u_idx in range(len(unique_users)):
        scores = np.array(user_to_scores[u_idx])
        post_indices = np.array(user_to_postidx[u_idx])

        if len(scores) == 0:
            topk_scores.append(np.array([]))
            topk_indices.append(np.array([]))
            continue

        # Sort descending
        sort_idx = np.argsort(-scores)

        topk = sort_idx[:K]
        topk_scores.append(scores[topk])
        topk_indices.append(post_indices[topk])

    # ---- 5. Construct new dict ----
    new_dict = {
        "user_int_ids": unique_users,
        "post_int_ids": unique_posts,
        "topk_scores": np.array(topk_scores),
        "topk_indices": np.array(topk_indices)
    }

    return new_dict

def _decode_ids(encoded_ids, decoder, to_numpy=False):
    ids = [decoder[id] for id in encoded_ids]
    return np.array(ids) if to_numpy else ids


def backfill_ranked_posts(posts, two_tower_posts, pooled_posts, top_k):
    merged_posts = []
    seen_posts = set()

    for source in (posts or [], two_tower_posts or [], pooled_posts or []):
        for post_id in source:
            if post_id not in seen_posts:
                merged_posts.append(post_id)
                seen_posts.add(post_id)
            if len(merged_posts) >= top_k:
                return merged_posts[:top_k]

    merged_posts = merged_posts[:top_k]
    if len(merged_posts) == top_k or not merged_posts:
        return merged_posts

    # Keep ranking/fallback order unchanged and only pad to enforce exact top_k length.
    missing = top_k - len(merged_posts)
    repeats = (missing // len(merged_posts)) + 1
    padded_tail = (merged_posts * repeats)[:missing]
    return merged_posts + padded_tail


def retrieve_k_posts_per_user(
    spark,
    model_res,
    decoder_uid,
    decoder_pid,
    top_k,
    table_name,
    fallback_table_name,
    pooled_candidates_table_name,
    run_date,
    chunk_size=5000,
    shard_id=0,
    num_shards=1,
):
    uids = model_res['user_int_ids']
    pids = model_res['post_int_ids']
    top_ind = model_res['topk_indices']

    uids = _decode_ids(uids, decoder_uid)
    pids = _decode_ids(pids, decoder_pid, to_numpy=True)

    backfill_udf = F.udf(
        lambda posts, two_tower_posts, pooled_posts: backfill_ranked_posts(posts, two_tower_posts, pooled_posts, top_k),
        ArrayType(StringType())
    )
    empty_post_array = F.array().cast(ArrayType(StringType()))
    fallback_two_tower_df = (
        spark.table(fallback_table_name)
    )
    fallback_two_tower_df = apply_partition_date_filter_if_exists(fallback_two_tower_df, run_date)
    fallback_two_tower_df = apply_shard_filter(fallback_two_tower_df, shard_id, num_shards)
    fallback_two_tower_df = fallback_two_tower_df.select("user_id", F.col("post_id").alias("two_tower_post_id"))

    fallback_pooled_df = (
        spark.table(pooled_candidates_table_name)
    )
    fallback_pooled_df = apply_partition_date_filter_if_exists(fallback_pooled_df, run_date)
    fallback_pooled_df = apply_shard_filter(fallback_pooled_df, shard_id, num_shards)
    fallback_pooled_df = (
        fallback_pooled_df
        .select("user_id", "post_id")
        .groupBy("user_id")
        .agg(F.collect_list("post_id").alias("pooled_post_id"))
    )
    fallback_user_df = (
        fallback_two_tower_df.select("user_id")
        .union(fallback_pooled_df.select("user_id"))
        .distinct()
    )

    # define schema
    schema = StructType([
        StructField("user_id", StringType(), False),
        StructField("post_id", ArrayType(StringType()), False)
    ])

    # Process data in chunks to avoid memory issues
    wrote_any_rows = False
    for i in range(0, len(uids), chunk_size):
        chunk_data = []
        end_idx = min(i + chunk_size, len(uids))

        for j in range(i, end_idx):
            user_id = uids[j]
            indices = top_ind[j]
            posts = pids[np.asarray(indices, dtype=np.int64)[:top_k]].tolist()
            chunk_data.append((user_id, posts))

        chunk_df = spark.createDataFrame(chunk_data, schema=schema)
        chunk_df = (
            chunk_df
            .join(fallback_two_tower_df, on="user_id", how="left")
            .join(fallback_pooled_df, on="user_id", how="left")
            .withColumn(
                "post_id",
                backfill_udf(
                    F.col("post_id"),
                    F.coalesce(F.col("two_tower_post_id"), empty_post_array),
                    F.coalesce(F.col("pooled_post_id"), empty_post_array)
                )
            )
            .drop("two_tower_post_id", "pooled_post_id")
        )

        invalid_post_count = chunk_df.filter(F.size("post_id") != top_k).count()
        if invalid_post_count > 0:
            raise RuntimeError(
                f"Failed to build exactly {top_k} posts for {invalid_post_count} users in ranking output chunk starting at index {i}"
            )

        chunk_df = chunk_df.withColumn("partition_date", F.lit(run_date))

        if not wrote_any_rows:
            chunk_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
            wrote_any_rows = True
        else:
            chunk_df.write.mode("append").saveAsTable(table_name)

    ranked_user_ids_df = spark.createDataFrame(
        [(user_id,) for user_id in uids],
        StructType([StructField("user_id", StringType(), False)])
    )

    missing_users_df = (
        fallback_user_df
        .join(ranked_user_ids_df, on="user_id", how="left_anti")
        .join(fallback_two_tower_df, on="user_id", how="left")
        .join(fallback_pooled_df, on="user_id", how="left")
        .withColumn(
            "post_id",
            backfill_udf(
                empty_post_array,
                F.coalesce(F.col("two_tower_post_id"), empty_post_array),
                F.coalesce(F.col("pooled_post_id"), empty_post_array)
            )
        )
        .drop("two_tower_post_id", "pooled_post_id")
    )

    missing_invalid_post_count = missing_users_df.filter(F.size("post_id") != top_k).count()
    if missing_invalid_post_count > 0:
        raise RuntimeError(
            f"Failed to build exactly {top_k} posts for {missing_invalid_post_count} fallback-only users"
        )

    missing_users_df = missing_users_df.withColumn("partition_date", F.lit(run_date))
    if wrote_any_rows:
        missing_users_df.write.mode("append").saveAsTable(table_name)
    else:
        missing_users_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

    return None

In [0]:
# write retrieval results to table
model_res = load_and_merge_results(result_dir)
top_k = model_config['top_k_posts']
table_write_chunk_size = model_config['table_write_chunk_size']
results = transform_dict(model_res, top_k)

In [0]:
uid_decoder, pid_decoder = load_decoders(result_dir)
fallback_table_name = db+'.'+table_names['two_tower_retrieval']
pooled_candidates_table_name = db+'.'+table_names['all_retrieval_results']
ranking_output_table_name = (
    db + '.' + table_names['ranking_results'] + f"_parallel_temp_{ranking_shard_id}"
    if ranking_num_shards > 1
    else db + '.' + table_names['ranking_results']
)
print("Write results to table:")
retrieve_k_posts_per_user(spark,
                          results,
                          uid_decoder,
                          pid_decoder,
                          top_k=top_k,
                          table_name=ranking_output_table_name,
                          fallback_table_name=fallback_table_name,
                          pooled_candidates_table_name=pooled_candidates_table_name,
                          run_date=run_date,
                          chunk_size=table_write_chunk_size,
                          shard_id=ranking_shard_id,
                          num_shards=ranking_num_shards)